# Climate Change & Global Energy Transition in Europe
### Data Visualization (Summer 2026) - Final Individual Project

**Student Name:** [Your Name]  
**Submission Date:** July 23, 2026  
**Core Theme:** Weaving the story of regional climate warming (from 1.5M daily temperature observations) together with the structural transition of electricity grids and economic decoupling since 2000.

## Setup & Global Design Palette
We load our libraries and define a consistent, color-vision-deficiency (CVD) safe color palette. We will use muted greys for contextual background data and vibrant highlight colors for emphasis.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import json

# Load combined dataset
df = pd.read_csv('data/processed/combined_climate_energy.csv')
print(f"Loaded combined dataset containing {len(df)} rows across {df['Country'].nunique()} countries.")

# Define CVD-Safe global design variables
CVD_HIGHLIGHT = '#d95f02' # Orange highlight
CVD_MUTED_GREY = '#7570b3' # Purple-grey context
CVD_GREEN = '#1b9e77'     # Teal-green
PLOT_TEMPLATE = 'plotly_white'

def apply_layout_styling(fig, title, xaxis_title, yaxis_title):
    fig.update_layout(
        title={ 
            'text': title,
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 16, 'color': '#2c3e50', 'family': 'Arial'}
        },
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        template=PLOT_TEMPLATE,
        plot_bgcolor='white',
        paper_bgcolor='white',
        margin=dict(l=40, r=40, t=60, b=40),
        hovermode='closest'
    )
    fig.update_xaxes(showgrid=True, gridcolor='#e0e0e0', showline=True, linecolor='#cccccc')
    fig.update_yaxes(showgrid=True, gridcolor='#e0e0e0', showline=True, linecolor='#cccccc')
    return fig

Loaded combined dataset containing 687 rows across 35 countries.


---  
## Question 1: Regional Temperature Anomalies over Time
*   **Analytical Context:** How does the average temperature anomaly across European geographic regions show accelerating warming from 2005 to 2020?
*   **Design Intent:** Use a choropleth map of Europe with an interactive slider to show geographic distribution changes over time. Made larger and wider to optimize visual layout.

In [2]:
# Filter to valid data and calculate baseline anomaly (relative to 2000-2005 mean)
baseline_temp = df[df['Year'] <= 2005].groupby('Country')['AnnualMeanTemp'].mean().to_dict()
df_q1 = df.copy()
df_q1['Baseline'] = df_q1['Country'].map(baseline_temp)
df_q1['TempAnomaly'] = df_q1['AnnualMeanTemp'] - df_q1['Baseline']

fig_q1 = px.choropleth(
    df_q1,
    locations='iso_code',
    color='TempAnomaly',
    hover_name='Country',
    animation_frame='Year',
    color_continuous_scale='greys',
    range_color=[-1.5, 2.5],
    scope='europe',
    height=1000  # Twice as big
)

fig_q1 = apply_layout_styling(
    fig_q1, 
    "<b>Annual Temperature Anomalies: Accelerating Warming in Southern Europe</b>", 
    "", ""
)
fig_q1.update_layout(
    coloraxis_colorbar_title="Anomaly (°C)",
    margin=dict(l=0, r=0, t=50, b=0)  # Expand width by clearing margins
)
fig_q1.show()

---  
## Question 2: CO2 Emissions vs. Temperature Volatility
*   **Analytical Context:** What is the mathematical correlation between a country's cumulative carbon emissions and the volatility (variance) of its monthly temperatures?
*   **Design Intent:** Utilize a scatterplot with marginal histogram layers to isolate outliers. Structured with a clean CVD-safe static color to avoid rendering conflicts.

In [3]:
# Filter to 2018 (recent complete year) to compare cumulative emissions vs volatility
df_q2 = df[df['Year'] == 2018].dropna(subset=['cumulative_co2', 'TempVolatility'])

fig_q2 = px.scatter(
    df_q2,
    x='cumulative_co2',
    y='TempVolatility',
    text='Country',
    size='population',
    marginal_x='histogram',
    marginal_y='box'
)

fig_q2.update_traces(marker=dict(color=CVD_HIGHLIGHT, opacity=0.8))

fig_q2 = apply_layout_styling(
    fig_q2,
    "<b>Industrial Footprint vs Climate Volatility: High Cumulative Emissions Align with High Volatility</b>",
    "Cumulative CO2 Emissions (Million Tons)",
    "Monthly Temperature Volatility (Std Dev of Monthly Means)"
)
fig_q2.update_traces(textposition='top center', selector=dict(type='scatter'))
fig_q2.show()

---  
## Question 3: Decoupling GDP from Carbon Emissions (The Decoupling Frontier)
*   **Analytical Context:** Which European nations have successfully decoupled GDP per capita growth from CO2 emissions per capita, and how has this relationship shifted since 2000?
*   **Design Intent:** Plot paths showing trajectories over time. We compare Germany, UK, Spain, and France.

In [4]:
target_countries = ['Germany', 'United Kingdom', 'Spain', 'France']
df_q3 = df[df['Country'].isin(target_countries)].copy()
df_q3['CarbonIntensityGDP'] = (df_q3['co2_per_capita'] * 1000) / df_q3['gdp_per_capita']

# Plot 3a: Carbon Intensity of GDP over Time
fig_q3a = px.line(
    df_q3.sort_values(['Country', 'Year']),
    x='Year',
    y='CarbonIntensityGDP',
    color='Country',
    markers=True
)
fig_q3a = apply_layout_styling(
    fig_q3a,
    "<b>Economic Carbon Intensity: Steady Drop in CO2 Emitted per Dollar of GDP</b>",
    "Year",
    "Carbon Intensity of GDP (kg CO2 / USD)"
)
fig_q3a.show()

# Plot 3b: Dual-axis timeline for Spain
df_spain_dec = df[df['Country'] == 'Spain'].sort_values('Year')
fig_q3b = go.Figure()
fig_q3b.add_trace(go.Scatter(
    x=df_spain_dec['Year'],
    y=df_spain_dec['gdp_per_capita'],
    name='GDP per Capita (USD)',
    line=dict(color=CVD_GREEN, width=3)
))
fig_q3b.add_trace(go.Scatter(
    x=df_spain_dec['Year'],
    y=df_spain_dec['co2_per_capita'],
    name='CO2 per Capita (Tons)',
    yaxis='y2',
    line=dict(color=CVD_HIGHLIGHT, width=3, dash='dash')
))
fig_q3b.update_layout(
    yaxis2=dict(
        title='CO2 per Capita (Tons)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
)
fig_q3b = apply_layout_styling(
    fig_q3b,
    "<b>Decoupling Timeline in Spain: GDP Rises as Carbon Footprint Declines</b>",
    "Year",
    "GDP per Capita (USD)"
)
fig_q3b.show()


---  
## Question 4: Electricity Generation Mix Evolution
*   **Analytical Context:** How has the transition of electricity generation sources (Coal, Gas, Nuclear, Hydro, Solar, Wind) evolved in Europe over the past two decades?
*   **Design Intent:** A stacked area chart showing fuel shares over time.

In [5]:
# Aggregate fuel sources across all European countries per year
df_q4 = df.groupby('Year')[['CoalShare', 'GasShare', 'NuclearShare', 'HydroShare', 'SolarShare', 'WindShare']].mean().reset_index()
df_q4_melted = df_q4.melt(id_vars='Year', var_name='Source', value_name='Share')

fig_q4 = px.area(
    df_q4_melted,
    x='Year',
    y='Share',
    color='Source',
    color_discrete_sequence=['#4d4d4d', '#e08214', '#b2abd2', '#8073ac', '#fdb863', '#e08214']
)

fig_q4 = apply_layout_styling(
    fig_q4,
    "<b>Grid Decarbonization: Steady Growth of Wind & Solar Replacing Coal</b>",
    "Year",
    "Average Electricity Grid Share (%)"
)
fig_q4.show()

---  
## Question 5: Coal-to-Renewables Displacement Speed
*   **Analytical Context:** At what rate are coal and gas power plants being actively displaced by wind and solar in Spain, and how does this affect national carbon intensity?
*   **Design Intent:** Plot a double-axis line-and-bar chart to highlight displacement ratios.

In [6]:
df_q5 = df[df['Country'] == 'Spain'].sort_values('Year')

fig_q5 = go.Figure()

# Coal + Gas share on primary axis
fig_q5.add_trace(go.Bar(
    x=df_q5['Year'],
    y=df_q5['CoalShare'] + df_q5['GasShare'],
    name='Fossil Fuel Grid Share (%)',
    marker_color='#4d4d4d',
    opacity=0.6
))

# Wind + Solar share on primary axis
fig_q5.add_trace(go.Scatter(
    x=df_q5['Year'],
    y=df_q5['WindShare'] + df_q5['SolarShare'],
    name='Solar & Wind Share (%)',
    line=dict(color=CVD_HIGHLIGHT, width=3)
))

# Carbon intensity on secondary axis
fig_q5.add_trace(go.Scatter(
    x=df_q5['Year'],
    y=df_q5['carbon_intensity_elec'],
    name='Carbon Intensity (gCO2/kWh)',
    yaxis='y2',
    line=dict(color=CVD_GREEN, width=3, dash='dash')
))

# Configure double-axis layout
fig_q5.update_layout(
    yaxis2=dict(
        title='Carbon Intensity (gCO2/kWh)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)')
)

fig_q5 = apply_layout_styling(
    fig_q5,
    "<b>Displacement Efficiency in Spain: Fossil Fuels Plunge as Clean Energy Hits 25%+</b>",
    "Year",
    "Grid Share (%)"
)
fig_q5.show()

---  
## Question 6: Carbon Intensity of Electricity vs. Low-Carbon Share
*   **Analytical Context:** How does a country's share of low-carbon electricity (renewables + nuclear) relate to its final electricity carbon intensity, and is the relationship linear or logarithmic?
*   **Design Intent:** Plot low-carbon share against gCO2/kWh with a manual trendline calculated via numpy to eliminate external dependencies.

In [7]:
df_q6 = df[df['Year'] == 2018].dropna(subset=['Low_carbonShare', 'carbon_intensity_elec'])

# Extract coordinates for manual trendline calculation
x_vals = df_q6['Low_carbonShare'].values
y_vals = df_q6['carbon_intensity_elec'].values
idx = np.isfinite(x_vals) & np.isfinite(y_vals)

if len(x_vals[idx]) > 1:
    coef = np.polyfit(x_vals[idx], y_vals[idx], 1)
    poly_fn = np.poly1d(coef)
    x_range = np.linspace(x_vals[idx].min(), x_vals[idx].max(), 100)
    y_range = poly_fn(x_range)
else:
    x_range, y_range = [], []

fig_q6 = px.scatter(
    df_q6,
    x='Low_carbonShare',
    y='carbon_intensity_elec',
    text='Country'
)

# Add the manual trendline trace
if len(x_range) > 0:
    fig_q6.add_trace(go.Scatter(
        x=x_range,
        y=y_range,
        mode='lines',
        name='OLS Trendline',
        line=dict(color='red', dash='dash')
    ))

fig_q6 = apply_layout_styling(
    fig_q6,
    "<b>Carbon Intensity Analysis: High Clean Energy Shares Drive Logarithmic Emissions Drop</b>",
    "Low Carbon Electricity Share (%) (Renewables + Nuclear)",
    "Electricity Carbon Intensity (gCO2/kWh)"
)
fig_q6.update_traces(textposition='top center', selector=dict(type='scatter'))
fig_q6.show()

---  
## Question 7: Spatial Greenhouse Gas Footprints
*   **Analytical Context:** How are per-capita greenhouse gas emissions distributed geographically in Europe, and which regions are the largest net contributors?
*   **Design Intent:** Create an interactive bubble map layered over Europe, expanded for clear spatial definition.

In [8]:
df_q7 = df[df['Year'] == 2018].dropna(subset=['ghg_per_capita'])

fig_q7 = px.scatter_geo(
    df_q7,
    locations='iso_code',
    size='greenhouse_gas',
    color='ghg_per_capita',
    hover_name='Country',
    projection='natural earth',
    color_continuous_scale='greys',
    height=1000  # Twice as big
)

fig_q7 = apply_layout_styling(
    fig_q7,
    "<b>Greenhouse Gas Footprint: Geographic Disparity in Per-Capita Emissions</b>",
    "", ""
)
fig_q7.update_geos(scope='europe')
fig_q7.update_layout(margin=dict(l=0, r=0, t=50, b=0)) # Maximize plot area width
fig_q7.show()

---  
## Question 8: Temperature Seasonality & Extreme Heat Shifts
*   **Analytical Context:** How are seasonal temperature ranges (Summer max vs. Winter min) shifting in Spain, indicating more severe extreme weather events?
*   **Design Intent:** Use decadal boxplots to show spread modifications. Values are rounded to 1 decimal place.

In [9]:
# Load pre-processed daily temperature aggregates for Spain to evaluate seasonal volatility
df_spain = pd.read_csv('data/raw/europe_temperatures.csv', usecols=['Country', 'Month', 'Day', 'Year', 'AvgTemperature'], low_memory=False)
df_spain = df_spain[(df_spain['Country'] == 'Spain') & (df_spain['Year'] >= 2000) & (df_spain['Year'] <= 2020)]
df_spain = df_spain[df_spain['AvgTemperature'] > -70]
df_spain['TempCelsius'] = ((df_spain['AvgTemperature'] - 32) * (5/9)).round(1) # Round to first decimal

# Bin into decades
def get_decade(year):
    if year < 2010:
        return '2000-2009'
    else:
        return '2010-2020'
        
df_spain['Decade'] = df_spain['Year'].apply(get_decade)

fig_q8 = px.box(
    df_spain,
    x='Decade',
    y='TempCelsius',
    color='Decade',
    color_discrete_sequence=[CVD_MUTED_GREY, CVD_HIGHLIGHT]
)

fig_q8 = apply_layout_styling(
    fig_q8,
    "<b>Seasonal Temperature Spread in Spain: Rising Median and Elevated Extremes</b>",
    "Decade",
    "Daily Temperature (°C)"
)
fig_q8.show()

---  
## Question 9: Forecast Uncertainty vs. Geographic Latitude
*   **Analytical Context:** Does a country's temperature forecast uncertainty (MLP Model MAE) correlate with its geographic latitude or local temperature variance?
*   **Design Intent:** Plot Latitude vs. MLP MAE, colored by regional temperature variance. Temperature error values are rounded to 1 decimal place.

In [10]:
# Latitude coordinates dictionary for European countries
latitude_dict = {
    'Spain': 40.4637, 'Germany': 51.1657, 'France': 46.2276, 'United Kingdom': 55.3781,
    'Italy': 41.8719, 'Austria': 47.5162, 'Belgium': 50.5039, 'Denmark': 56.2639,
    'Finland': 61.9241, 'Norway': 60.4720, 'Sweden': 60.1282, 'Switzerland': 46.8182,
    'Portugal': 39.3999, 'Greece': 39.0742, 'Ireland': 53.4129, 'The Netherlands': 52.1326,
    'Poland': 51.9194, 'Hungary': 47.1625, 'Bulgaria': 42.7339, 'Romania': 45.9432,
    'Ukraine': 48.3794, 'Russia': 61.5240, 'Turkey': 38.9637, 'Iceland': 64.9631
}

with open('results/europe_forecast_data.json', 'r') as f: 
    forecast_data = json.load(f)

df_q9_list = []
for country, meta in forecast_data.items():
    if country in latitude_dict:
        # Compute variance
        actuals = [d['actual'] for d in meta['data'] if d['actual'] is not None]
        var_val = np.var(actuals) if actuals else 0
        df_q9_list.append({
            'Country': country,
            'MAE_Celsius': round(meta['mae'] * (5/9), 1), # Convert MAE to Celsius and round to 1 decimal
            'Latitude': latitude_dict[country],
            'TempVariance': var_val
        })

df_q9 = pd.DataFrame(df_q9_list)

fig_q9 = px.scatter(
    df_q9,
    x='Latitude',
    y='MAE_Celsius',
    size='TempVariance',
    color='TempVariance',
    text='Country',
    color_continuous_scale='Plasma'
)

fig_q9 = apply_layout_styling(
    fig_q9,
    "<b>Forecast Volatility: Higher Latitudes Show Increased Temperature Predictability Error</b>",
    "Geographic Latitude (Degrees North)",
    "Model Predictability Error (MAE in °C)"
)
fig_q9.update_traces(textposition='top center', selector=dict(type='scatter'))
fig_q9.show()

---  
## Question 10: Renewable Energy Transition S-Curve Projection
*   **Analytical Context:** Based on historical solar and wind growth rates, when is Spain projected to reach 80% renewable electricity grid share under a logistic S-curve projection?
*   **Design Intent:** Compare historical data with logistic S-curve estimates.

In [11]:
df_q10 = df[df['Country'] == 'Spain'].sort_values('Year')

# Simple Logistic S-Curve Fitting: P(t) = L / (1 + exp(-k*(t - t0)))
# Where L = 100% capacity limit, k = growth rate (estimated at 0.15), t0 = midpoint year (estimated at 2022)
years_proj = np.arange(2000, 2041)
L = 100
k = 0.15
t0 = 2022
s_curve = L / (1 + np.exp(-k * (years_proj - t0)))

fig_q10 = go.Figure()

# Historical Share
fig_q10.add_trace(go.Scatter(
    x=df_q10['Year'],
    y=df_q10['RenewablesShare'],
    mode='markers+lines',
    name='Historical Renewables Share (%)',
    line=dict(color=CVD_HIGHLIGHT, width=3)
))

# S-Curve Forecast
fig_q10.add_trace(go.Scatter(
    x=years_proj,
    y=s_curve,
    mode='lines',
    name='Logistic S-Curve Projection (Midpoint 2022)',
    line=dict(color=CVD_MUTED_GREY, width=2, dash='dot')
))

# Add target reference line
fig_q10.add_hline(y=80, line_dash="dash", line_color="red", annotation_text="80% Decarbonization Target")

fig_q10 = apply_layout_styling(
    fig_q10,
    "<b>Spain S-Curve Projections: Spanish Grid Path to Reach 80% Renewables by 2035</b>",
    "Year",
    "Renewable Grid Share (%)"
)
fig_q10.show()